# Phase 4 — Agent A1 : Loader / Validator
**YouTube Quality Analyzer** | Phase 4 — Implémentation

Ce notebook démontre et valide l'agent A1 (`agents/a1_loader.py`) :
- Chargement du CSV de commentaires pré-collectés
- Validation de la colonne `text` (obligatoire)
- Gestion des colonnes optionnelles (`video_id`, `author_likes`, `reply_count`)
- Filtrage des lignes à texte vide

> **Prérequis** : `pip install pandas` — aucun LLM requis.

In [15]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # rend le projet importable

import pandas as pd
from agents.a1_loader import a1_loader, REQUIRED_COLUMNS, OPTIONAL_COLUMNS

print("Colonnes obligatoires :", REQUIRED_COLUMNS)
print("Colonnes optionnelles :", OPTIONAL_COLUMNS)

Colonnes obligatoires : {'text'}
Colonnes optionnelles : {'author_likes', 'reply_count', 'video_id'}


## 1. Création d'un CSV de test

In [16]:
import tempfile, pathlib, pandas as pd

# CSV minimal (colonne text obligatoire + optionnelles)
rows = [
    {"text": "This is a great tutorial on machine learning!", "video_id": "abc123", "author_likes": 12, "reply_count": 3},
    {"text": "Very informative, learned a lot about neural networks.", "video_id": "abc123", "author_likes": 8,  "reply_count": 1},
    {"text": "Excellent explanation of gradient descent.", "video_id": "abc123", "author_likes": 5,  "reply_count": 0},
    {"text": "",                                                              "video_id": "abc123", "author_likes": 0,  "reply_count": 0},  # ligne vide → filtrée
    {"text": "Could you cover backpropagation next?",             "video_id": "abc123", "author_likes": 2,  "reply_count": 0},
]

tmp = tempfile.mkdtemp()
csv_path = str(pathlib.Path(tmp) / "comments.csv")
pd.DataFrame(rows).to_csv(csv_path, index=False)
print(f"CSV créé : {csv_path}")
print(pd.read_csv(csv_path))

CSV créé : C:\Users\MASSO\AppData\Local\Temp\tmpb3m46ctz\comments.csv
                                                text video_id  author_likes  \
0      This is a great tutorial on machine learning!   abc123            12   
1  Very informative, learned a lot about neural n...   abc123             8   
2         Excellent explanation of gradient descent.   abc123             5   
3                                                NaN   abc123             0   
4              Could you cover backpropagation next?   abc123             2   

   reply_count  
0            3  
1            1  
2            0  
3            0  
4            0  


## 2. Exécution de A1 — cas nominal

In [17]:
result = a1_loader({"csv_path": csv_path})

print(f"Commentaires chargés : {len(result['raw_comments'])}")
for i, c in enumerate(result["raw_comments"]):
    print(f"  [{i}] text={c['text']!r:50s}  video_id={c.get('video_id')}  likes={c.get('author_likes')}")

{"level": "INFO", "message": "a1_loader: renaming columns {'reply_count': 'reply_count'}", "logger": "a1_loader"}
{"level": "INFO", "message": "a1_loader: loaded 4 comments from C:\\Users\\MASSO\\AppData\\Local\\Temp\\tmpb3m46ctz\\comments.csv", "logger": "a1_loader"}
Commentaires chargés : 4
  [0] text='This is a great tutorial on machine learning!'     video_id=abc123  likes=12
  [1] text='Very informative, learned a lot about neural networks.'  video_id=abc123  likes=8
  [2] text='Excellent explanation of gradient descent.'        video_id=abc123  likes=5
  [3] text='Could you cover backpropagation next?'             video_id=abc123  likes=2


## 3. Cas d'erreur

In [18]:
# Cas 1 : fichier inexistant
err1 = a1_loader({"csv_path": "/nonexistent/path/file.csv"})
print("Cas 1 — fichier manquant:", err1.get("errors"))

# Cas 2 : csv_path vide
err2 = a1_loader({"csv_path": ""})
print("Cas 2 — csv_path vide:", err2.get("errors"))

# Cas 3 : colonne obligatoire manquante
import tempfile, pathlib, pandas as pd
tmp2 = tempfile.mkdtemp()
bad_csv = str(pathlib.Path(tmp2) / "bad.csv")
pd.DataFrame([{"author": "Bob", "likes": 5}]).to_csv(bad_csv, index=False)
err3 = a1_loader({"csv_path": bad_csv})
print("Cas 3 — colonne 'text' absente:", err3.get("errors"))

# Cas 4 : bypass (raw_comments pré-chargé)
bypass = a1_loader({"raw_comments": [{"text": "pre-loaded"}], "csv_path": bad_csv})
print("Cas 4 — bypass raw_comments:", bypass)  # doit retourner {}

Cas 1 — fichier manquant: ['a1_loader: file not found — /nonexistent/path/file.csv']
Cas 2 — csv_path vide: ['a1_loader: csv_path is missing from state.']
{"level": "INFO", "message": "a1_loader: renaming columns {'likes': 'author_likes'}", "logger": "a1_loader"}
Cas 3 — colonne 'text' absente: ["a1_loader: missing required columns: {'text'}"]
{"level": "INFO", "message": "a1_loader: raw_comments already provided, skipping CSV load.", "logger": "a1_loader"}
Cas 4 — bypass raw_comments: {}


## Résumé A1

| Comportement | Résultat |
|---|---|
| CSV valide avec colonnes optionnelles | Tous les records chargés avec métadonnées |
| Ligne texte vide filtrée | Oui |
| CSV manquant | Erreur dans `errors[]` |
| Colonne `text` absente | Erreur dans `errors[]` |
| `raw_comments` pré-chargé | Bypass silencieux (retourne `{}`) |

> **Prochaine étape** : fournir `data/raw/comments.csv` pour exécuter A1 sur données réelles → tag `v0.1.0`

---

# Tests Agent A0 — Collector (PRD v1.1)

Ce notebook valide également l'agent A0 (`agents/a0_collector.py`) :
- Extraction du `video_id` depuis tous les formats d'URL (FR-74)
- Mode fallback CSV (sans clé API YouTube)
- Gestion des erreurs bloquantes (vidéo introuvable, commentaires désactivés)
- Récupération de la transcription (FR-84)
- Intégration cache Q&A (FR-86)

## A0 — 1. Extraction du video_id (FR-74)

Valide les 4 formats d'URL supportés + ID direct.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from agents.a0_collector import extract_video_id

test_cases = [
    ("URL standard",  "https://www.youtube.com/watch?v=dQw4w9WgXcQ", "dQw4w9WgXcQ"),
    ("URL courte",    "https://youtu.be/dQw4w9WgXcQ",                  "dQw4w9WgXcQ"),
    ("Shorts",        "https://www.youtube.com/shorts/dQw4w9WgXcQ",    "dQw4w9WgXcQ"),
    ("ID direct",     "dQw4w9WgXcQ",                                    "dQw4w9WgXcQ"),
]

print("=" * 60)
all_ok = True
for label, url, expected in test_cases:
    try:
        result = extract_video_id(url)
        status = "✓" if result == expected else "✗"
        if result != expected:
            all_ok = False
        print(f"  {status} {label:15s} → {result}")
    except ValueError as e:
        print(f"  ✗ {label:15s} → ERREUR: {e}")
        all_ok = False

# Cas d'erreur attendu
try:
    extract_video_id("https://example.com/not-youtube")
    print("  ✗ URL invalide → aurait dû lever ValueError")
    all_ok = False
except ValueError:
    print("  ✓ URL invalide   → ValueError levée correctement")

print("=" * 60)
print("Résultat global :", "PASS" if all_ok else "FAIL")

## A0 — 2. Mode fallback CSV (sans clé API)

Valide que A0 bascule sur le CSV pré-collecté quand `YOUTUBE_API_KEY` est absente (FR-80).  
Utilise le CSV de test créé pour A1 (section précédente du notebook).

In [ ]:
import tempfile, pathlib, pandas as pd, os
from agents.a0_collector import a0_collector

# Créer un CSV de fallback avec video_id connu
VIDEO_ID_FALLBACK = "testVideoId01"
rows_fallback = [
    {"video_id": VIDEO_ID_FALLBACK, "text": f"Super commentaire {i}", "author": f"User{i}",
     "author_likes": i, "reply_count": 0, "published_at": "2024-01-01T00:00:00Z",
     "is_reply": False, "parent_id": "", "language_detected": ""}
    for i in range(1, 8)  # 7 commentaires > seuil minimum de 5
]

tmp_dir   = tempfile.mkdtemp()
csv_path  = str(pathlib.Path(tmp_dir) / "comments_raw.csv")
pd.DataFrame(rows_fallback).to_csv(csv_path, index=False)
print(f"CSV fallback créé : {csv_path}")

# Supprimer la clé API pour forcer le fallback
old_key = os.environ.pop("YOUTUBE_API_KEY", None)

# Patcher le chemin fallback
import agents.a0_collector as a0_mod
original_fallback = a0_mod._FALLBACK_CSV
a0_mod._FALLBACK_CSV = csv_path

result = a0_collector({"url_or_id": VIDEO_ID_FALLBACK})

# Restaurer
a0_mod._FALLBACK_CSV = original_fallback
if old_key:
    os.environ["YOUTUBE_API_KEY"] = old_key

print(f"\nsource         : {result.get('source')}")           # attendu : csv_fallback
print(f"video_id       : {result.get('video_id')}")
print(f"commentaires   : {len(result.get('raw_comments', []))}")  # attendu : 7
print(f"errors         : {result.get('errors')}")
assert result.get("source") == "csv_fallback", "source doit être csv_fallback"
assert len(result.get("raw_comments", [])) == 7,  "7 commentaires attendus"
print("\n✓ Fallback CSV fonctionne correctement (FR-80)")

## A0 — 3. Erreur INSUFFICIENT_COMMENTS (§5.3)

Valide que A0 s'arrête avec un code d'erreur explicite si moins de 5 commentaires collectés.

In [ ]:
import tempfile, pathlib, pandas as pd, os
import agents.a0_collector as a0_mod

VIDEO_ID_SPARSE = "sparseVideo01"

# CSV avec seulement 2 commentaires (< seuil min = 5)
rows_sparse = [
    {"video_id": VIDEO_ID_SPARSE, "text": f"Commentaire {i}", "author": f"U{i}",
     "author_likes": 0, "reply_count": 0, "published_at": "2024-01-01T00:00:00Z",
     "is_reply": False, "parent_id": "", "language_detected": ""}
    for i in range(1, 3)  # seulement 2
]

tmp_dir  = tempfile.mkdtemp()
csv_path = str(pathlib.Path(tmp_dir) / "sparse.csv")
pd.DataFrame(rows_sparse).to_csv(csv_path, index=False)

old_key = os.environ.pop("YOUTUBE_API_KEY", None)
original_fallback = a0_mod._FALLBACK_CSV
a0_mod._FALLBACK_CSV = csv_path

result = a0_collector({"url_or_id": VIDEO_ID_SPARSE})

a0_mod._FALLBACK_CSV = original_fallback
if old_key:
    os.environ["YOUTUBE_API_KEY"] = old_key

errors = result.get("errors", [])
print(f"errors : {errors}")
assert any("INSUFFICIENT_COMMENTS" in e for e in errors), "INSUFFICIENT_COMMENTS attendu"
assert "raw_comments" not in result or not result.get("raw_comments"), "Pas de raw_comments attendu"
print("✓ INSUFFICIENT_COMMENTS retourné correctement (§5.3)")

## A0 — 4. Cache Q&A (FR-86)

Valide que le cache stocke bien le rapport ET le `qa_context` après un pipeline complet.

In [ ]:
from api.cache import ReportCache

cache = ReportCache()

# Simuler un rapport stocké après pipeline
fake_report = {
    "video_id": "dQw4w9WgXcQ",
    "topic": "musique",
    "score_global": 72.0,
    "score_pertinence": 65.0,
    "score_final": 69.2,
    "quality_label": "Bon",
}
cache.set("dQw4w9WgXcQ", "musique", fake_report)

# Simuler le qa_context stocké par /analyze (FR-86)
fake_qa_ctx = {
    "transcript": [
        {"text": "Bonjour et bienvenue dans cette vidéo.", "start": 0.0, "duration": 3.5},
        {"text": "Aujourd'hui on parle de musique pop.", "start": 3.5, "duration": 4.0},
    ],
    "transcript_available": True,
    "top_comments": [
        {"comment_id": "c001", "text": "Excellente vidéo !", "score_a4": 0.82},
        {"comment_id": "c002", "text": "Très bonne analyse de la chanson.", "score_a4": 0.75},
    ],
    "video_title": "Rick Astley - Never Gonna Give You Up",
}
cache.set_qa_context("dQw4w9WgXcQ", fake_qa_ctx)

# Vérifications
assert cache.exists("dQw4w9WgXcQ", "musique"),     "exists() doit retourner True"
assert cache.has_qa_context("dQw4w9WgXcQ"),         "has_qa_context() doit retourner True"
assert not cache.exists("dQw4w9WgXcQ", "sport"),    "topic différent → cache MISS"
assert not cache.has_qa_context("autreVideo000"),   "autre video_id → pas de qa_context"

retrieved = cache.get("dQw4w9WgXcQ", "musique")
qa_ctx    = cache.get_qa_context("dQw4w9WgXcQ")

print(f"Rapport récupéré  : score_global={retrieved['score_global']}")
print(f"qa_context        : transcript_available={qa_ctx['transcript_available']}")
print(f"  segments        : {len(qa_ctx['transcript'])}")
print(f"  top_comments    : {len(qa_ctx['top_comments'])}")

# Vérification clé de cache (FR-82) : topic normalisé
assert cache._key("dQw4w9WgXcQ", "  Musique  ") == "dQw4w9WgXcQ::musique"
print("\n✓ Cache rapport + qa_context fonctionne (FR-82, FR-83, FR-86)")

## A0 — 5. Module Q&A — Construction du prompt et parsing

Valide la construction du prompt grounded et le parsing JSON de la réponse LLM sans invoquer de LLM réel.

In [ ]:
from api.qa import (
    _build_transcript_context,
    _build_comments_context,
    _build_history_context,
    _build_prompt,
    _parse_llm_response,
)

# ── Test construction transcription ──────────────────────────────────────────
transcript = [
    {"text": "Bonjour et bienvenue.", "start": 0.0,  "duration": 3.0},
    {"text": "Aujourd'hui, les intégrales.", "start": 3.0, "duration": 4.0},
    {"text": "La formule principale est ∫f(x)dx.", "start": 7.0, "duration": 5.0},
]
transcript_ctx = _build_transcript_context(transcript)
assert "[00:00] Bonjour" in transcript_ctx
assert "[00:07]" in transcript_ctx
print("✓ Transcription horodatée correctement construite")

# ── Test construction commentaires ────────────────────────────────────────────
top_comments = [
    {"comment_id": "c001", "text": "Super explication des intégrales !", "score_a4": 0.85},
    {"comment_id": "c002", "text": "La démonstration à 00:07 est très claire.", "score_a4": 0.78},
]
comments_ctx = _build_comments_context(top_comments)
assert "[c001]" in comments_ctx and "[c002]" in comments_ctx
print("✓ Commentaires Q&A construits correctement")

# ── Test construction historique ──────────────────────────────────────────────
history = [
    {"role": "user",      "content": "De quoi parle la vidéo ?"},
    {"role": "assistant", "content": "La vidéo traite des intégrales."},
]
history_ctx = _build_history_context(history)
assert "Utilisateur" in history_ctx and "Assistant" in history_ctx
print("✓ Historique formaté correctement")

# ── Test assemblage prompt complet ────────────────────────────────────────────
prompt = _build_prompt(
    question="Quelle est la formule principale ?",
    transcript_ctx=transcript_ctx,
    comments_ctx=comments_ctx,
    history_ctx=history_ctx,
    video_title="Cours de maths — Intégrales",
    transcript_available=True,
)
assert "CONTEXTE — TRANSCRIPTION" in prompt
assert "CONTEXTE — COMMENTAIRES" in prompt
assert "HISTORIQUE DE CONVERSATION" in prompt
assert "QUESTION ACTUELLE" in prompt
print("✓ Prompt complet assemblé avec toutes les sections")

# ── Test parsing JSON LLM ─────────────────────────────────────────────────────
raw_json = """{
  "answer": "La formule principale est ∫f(x)dx.",
  "sources": [{"type": "transcript", "start": 7.0, "text": "La formule principale est ∫f(x)dx."}],
  "confidence": 0.91
}"""
parsed = _parse_llm_response(raw_json)
assert parsed["answer"] == "La formule principale est ∫f(x)dx."
assert len(parsed["sources"]) == 1
assert abs(parsed["confidence"] - 0.91) < 0.001
print("✓ Parsing JSON LLM réussi")

# ── Test parsing avec code block markdown ─────────────────────────────────────
raw_with_block = '```json\n{"answer": "Réponse test", "sources": [], "confidence": 0.5}\n```'
parsed2 = _parse_llm_response(raw_with_block)
assert parsed2["answer"] == "Réponse test"
print("✓ Parsing avec bloc ```json``` fonctionne")

print("\n✓ Module Q&A — construction prompt + parsing : PASS (FR-88, FR-89, FR-90)")

## Résumé A0 + Q&A

| Critère PRD | Exigence | Statut |
|---|---|---|
| FR-74 | extract_video_id — 4 formats URL | ✓ Testé |
| FR-80 | Fallback CSV si pas de clé API | ✓ Testé |
| §5.3 | INSUFFICIENT_COMMENTS si < 5 | ✓ Testé |
| FR-82 | Clé cache avec topic normalisé | ✓ Testé |
| FR-83 | Interface get/set/exists/qa_context | ✓ Testé |
| FR-86 | qa_context stocké en cache | ✓ Testé |
| FR-88 | Prompt grounding UNIQUEMENT contexte | ✓ Testé |
| FR-89 | Réponse avec `sources` citées | ✓ Testé |
| FR-90 | Historique max 5 tours | ✓ Testé |

**Fichiers impliqués :**
- `agents/a0_collector.py` — Agent A0 complet
- `api/cache.py` — Cache rapport + qa_context
- `api/qa.py` — Module Q&A grounded
- `api/routes.py` — `/analyze` branché sur A0 + endpoint `/ask`
- `api/schemas.py` — AskRequest / AskResponse
- `pipeline_state.py` — Champs A0 déclarés

> **Prochaine étape (avec clé API réelle)** : tester `a0_collector` sur une vraie URL YouTube publique → AC-13, AC-18

---

# Test réel — A0 avec clé YouTube API (AC-13)

Collecte les commentaires d'une vraie vidéo YouTube publique et valide le CSV produit.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

# Charger .env
from dotenv import load_dotenv
load_dotenv(dotenv_path=os.path.join("..", ".env"), override=True)

# Vérifier la clé
api_key = os.getenv("YOUTUBE_API_KEY", "")
print("YOUTUBE_API_KEY :", "✓ chargée" if api_key else "✗ absente")
print("OPENAI_API_KEY  :", "✓ chargée" if os.getenv("OPENAI_API_KEY") else "✗ absente")
print("LLM_BACKEND     :", os.getenv("LLM_BACKEND", "openai"))
print("LLM_MODEL       :", os.getenv("LLM_MODEL", "gpt-4o-mini"))

In [ ]:
from agents.a0_collector import a0_collector

# Vidéo YouTube publique avec commentaires actifs
# (changer l'URL si nécessaire)
TEST_URL = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"

print(f"Collecte en cours pour : {TEST_URL}")
print("(peut prendre jusqu'à 20s — pagination API)\n")

result = a0_collector({"url_or_id": TEST_URL, "topic": "musique"})

print(f"video_id             : {result.get('video_id')}")
print(f"source               : {result.get('source')}")
print(f"quota_used           : {result.get('quota_used')} unités")
print(f"commentaires         : {len(result.get('raw_comments', []))}")
print(f"transcript_available : {result.get('transcript_available')}")
print(f"transcript segments  : {len(result.get('transcript', []))}")
print(f"errors               : {result.get('errors')}")

if result.get("raw_comments"):
    print("\n── Aperçu des 3 premiers commentaires ──")
    for c in result["raw_comments"][:3]:
        print(f"  [{c['comment_id']}] {c['text'][:80]!r}")

## Test LLM OpenAI (AC-19)

Valide que le LLM répond correctement avec la clé OpenAI.

In [ ]:
from models.llm_loader import get_llm
from langchain_core.messages import HumanMessage, SystemMessage

llm = get_llm()
if llm is None:
    print("✗ LLM non chargé — vérifier OPENAI_API_KEY dans .env")
else:
    print(f"✓ LLM chargé : {type(llm).__name__}")
    resp = llm.invoke([
        SystemMessage(content="Tu es un assistant concis. Réponds en une phrase."),
        HumanMessage(content="Dis bonjour en français."),
    ])
    print(f"Réponse : {resp.content}")